# 冒烟测试

> 不依赖数据，依然是验证 M2 关于因子适应性的代码是否合理


In [1]:
import numpy as np
import pandas as pd

from src.evaluation.fitness import compute_forward_returns, make_fitness
from src.gp.engine import GPConfig, random_tree, run_gp
from src.gp.evaluator import to_wide

# 造迷你长表：5 只股票 × 60 天
dates = pd.date_range("2024-01-01", periods=60, freq="B")
codes = [f"s{i:02d}" for i in range(20)]
idx = pd.MultiIndex.from_product([dates, codes], names=["date", "code"])
rng = np.random.default_rng(0)
prices = pd.DataFrame(
    {
        c: rng.random(len(idx)) * 100 + 10
        for c in ["open", "high", "low", "close", "volume", "amount"]
    },
    index=idx,
)

wide = to_wide(prices)
fwd = compute_forward_returns(wide["close"], period=5)
assert fwd.iloc[-5:].isna().all().all()  # 最后 5 行应全 NaN（无未来数据）

fitness_fn = make_fitness(wide, fwd)

# 单棵树：分数应是 [0, ~] 的有限值（随机数据下接近 0 很正常）
tree = random_tree(max_depth=4, min_depth=2)
score = fitness_fn(tree)
print("单棵适应度:", score, "| 表达式:", tree)
assert np.isfinite(score)

# 跑一个迷你进化（小种群短代数），确认整条链路通
results = run_gp(fitness_fn, GPConfig(population_size=30, generations=5, max_depth=4, min_depth=2))
print("最佳:", results[0][1], results[0][0])


单棵适应度: 2.944170851065453 | 表达式: div(rank(ts_rank(log(close))), sub(neg(ts_rank(high)), log(div(high, high))))
Gen   0 | best=1.9726 | mean=0.1991 | expr=neg(neg(sub(rank(close), add(amount, close))))
Gen   1 | best=2.5175 | mean=0.3710 | expr=sub(ts_rank(rank(ts_mean(high))), mul(volume, close))
Gen   2 | best=2.5175 | mean=1.2065 | expr=sub(ts_rank(rank(ts_mean(high))), mul(volume, close))
Gen   3 | best=4.3691 | mean=1.8473 | expr=neg(close)
Gen   4 | best=4.3691 | mean=2.5289 | expr=neg(close)
最佳: 4.369127745425523 neg(close)
